In [26]:
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import requests


In [6]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_api_url = "https://api.anthropic.com/v1/"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_api_url)

In [63]:
def fetch_website_contents(url: str) -> str | None:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title + "\n\n" + text

In [64]:
def fetch_wikipedia_page(subject:str) -> str | None:
    url = f"https://en.wikipedia.org/wiki/{subject.lower().strip().replace(' ', '_').replace("-", "_")}"
    return fetch_website_contents(url)

In [66]:
def summarize_text(text: str, model: str = "gpt-4o-mini") -> str | None:
    system_prompt = """You are a helpful assistant that summarizes text in a concise, structured and compelling way,
    ignoring text that might be navigation related. 
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Summarize the following text in a concise, structured and compelling way:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content

In [67]:
content = fetch_wikipedia_page("Paris")
if content:
    summary = summarize_text(content)
    print(summary)

# Paris Overview

## General Information
- **Capital of France**: Paris, often referred to as the "City of Light," is the largest city in France, with a city population of approximately 2.04 million and a metropolitan area population of around 13.2 million as of January 2026.
- **Geographic Location**: Situated on the Seine River in the Île-de-France region, Paris is the largest metropolitan area in the EU.

## History
- **Etymology**: The name "Paris" is derived from the Parisii, a Celtic tribe. It was first mentioned as Lutetia Parisiorum by Julius Caesar.
- **Historical Development**:
  - **Origins**: Established by the Parisii around the 3rd century BC.
  - **Middle Ages**: Became a political and cultural center by the end of the 12th century.
  - **Modern Era**: Underwent significant transformations during the Haussmann renovation in the 19th century, creating wide boulevards and iconic parks.

## Administration
- **Government Structure**: Paris is divided into 20 arrondissements,